# BDD Panel — RSE × Performance Financière (2021-2024)

**Format : panel long — 1 ligne = 1 SIREN × 1 année**

Toutes les variables des deux bases sont conservées. La sélection/suppression des variables problématiques sera effectuée manuellement lors de l'étape d'analyse statistique.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Chargement

In [2]:
fin_raw = pd.read_excel('bdd_final.xlsx', dtype={'SIREN': str})
rse_raw = pd.read_csv(
    'resultats_rse_scores_2021_2024.csv',
    sep=None, engine='python', encoding='utf-8-sig', dtype={'SIREN': str}
)

print(f'Financier brut : {fin_raw.shape} | {fin_raw["SIREN"].nunique()} SIREN')
print(f'RSE brut       : {rse_raw.shape} | {rse_raw["SIREN"].nunique()} SIREN')

Financier brut : (4964, 21) | 2221 SIREN
RSE brut       : (2221, 17) | 2221 SIREN


## 2. Nettoyage & préparation

In [3]:
# --- Financier ---
fin = fin_raw.rename(columns={
    "Dénomination sociale de l'entreprise": 'raison_sociale',
    "Nom commercial de l'entreprise": 'nom_commercial',
    'Date societe a mission': 'date_mission',
    'Date de création': 'date_creation',
    'Code NAF - APE': 'code_naf',
    'Libelle APE': 'libelle_naf',
    'Date de vote de la mission par les actionnaires': 'date_vote_mission',
    'Année': 'annee',
    "chiffre d'affaires (€)": 'ca',
    "chiffre d'affaires à l'export (€)": 'ca_export',
    'dettes financières (€)': 'dettes_fin',
    'fonds propres (€)': 'fonds_propres',
    "ratio d'endettement (gearing)": 'gearing',
    'rentabilité sur fonds propres (%)': 'roe_pct',
    'rentabilité économique (%)': 'roa_pct',
    "résultat d'exploitation (€)": 'ebit',
    'résultat net (€)': 'resultat_net',
    'Total Actif': 'total_actif'
})

# Toutes les variables financières conservées — le nettoyage sera fait manuellement
FIN_VARS = ['ca', 'ca_export', 'resultat_net', 'fonds_propres', 'dettes_fin', 'gearing', 'roe_pct', 'roa_pct', 'ebit', 'total_actif']
fin[FIN_VARS] = fin[FIN_VARS].apply(pd.to_numeric, errors='coerce')
fin['annee'] = fin['annee'].astype('Int64')

# --- RSE ---
rse = rse_raw.rename(columns={
    "Dénomination sociale de l'entreprise": 'raison_sociale',
    'Date societe a mission': 'date_mission',
    'Date de création': 'date_creation',
    'Code NAF - APE': 'code_naf',
    'Libelle APE': 'libelle_naf',
})

## 3. Préparation des scores RSE

In [4]:
THEMES = ['ENVIRONNEMENT', 'ETHIQUE', 'SOCIAL & DROITS HUMAINS', 'ACHATS RESPONSABLES']
SCORE_COLS = ['Score_RSE_2021', 'Score_RSE_2022', 'Score_RSE_2023', 'Score_RSE_2024']

# Métadonnées entreprise
meta = rse[['SIREN', 'raison_sociale', 'date_mission', 'code_naf', 'libelle_naf', 'Secteur_EcoVadis']].drop_duplicates('SIREN')

# Score global annuel (moyenne des 4 thèmes) + score moyen sur la période
score_global = (
    rse[rse['Type_de_RSE'].isin(THEMES)]
    .groupby('SIREN')[SCORE_COLS]
    .mean()
    .rename(columns=lambda c: c.replace('Score_RSE_', 'score_global_'))
    .reset_index()
)
score_global['score_rse_moy'] = score_global[['score_global_2021','score_global_2022','score_global_2023','score_global_2024']].mean(axis=1)

# Table RSE finale (par SIREN)
# score_global_XXXX déjà portés en long via score_annuel — on garde seulement score_rse_moy ici
rse_final = meta.merge(
    score_global[['SIREN', 'score_rse_moy']], on='SIREN'
)

rse_final.head(2)

,SIREN,raison_sociale,date_mission,code_naf,libelle_naf,Secteur_EcoVadis,score_rse_moy
0,948955356,BIASSA,2023-02-15,3811Z,Collecte des déchets non dangereux,Manufacturing Heavy,64.2500
1,383699048,RAMSAY SANTE,2023-02-13,6430Z,Fonds de placement et entités financières simi...,"Finance, Legal & Consulting",59.1500


## 4. Construction du panel long — 1 ligne = 1 SIREN × 1 année

In [5]:
ANNEES = [2021, 2022, 2023, 2024]

# Filtrer le financier sur 2021-2024 et merger avec les scores RSE annuels
fin_2124 = fin[fin['annee'].isin(ANNEES)].copy()

# Ajouter le score RSE de l'année correspondante (format long)
score_annuel = score_global.melt(
    id_vars='SIREN',
    value_vars=['score_global_2021','score_global_2022','score_global_2023','score_global_2024'],
    var_name='annee', value_name='score_rse'
)
score_annuel['annee'] = score_annuel['annee'].str.extract(r'(\d{4})').astype(int)

# Merge : financier × score annuel × métadonnées RSE
# On sélectionne explicitement les colonnes RSE pour éviter les doublons _x/_y
panel = (
    fin_2124
    .merge(score_annuel, on=['SIREN', 'annee'], how='inner')
    .merge(
        rse_final[['SIREN', 'raison_sociale', 'code_naf', 'libelle_naf',
                   'Secteur_EcoVadis', 'date_mission', 'score_rse_moy']],
        on='SIREN', how='left'
    )
)

# Variables normalisées
panel['rentabilite_nette'] = panel['resultat_net'] / panel['total_actif']   # ROA proxy
panel['tx_endettement']    = panel['dettes_fin']   / panel['total_actif']   # levier normalisé
panel['log_total_actif']   = np.log1p(panel['total_actif'].clip(lower=0))   # contrôle taille

# Suppression des colonnes redondantes / aberrantes
DROP_COLS = [
    # Éventuels doublons _x/_y si le merge crée des conflits
    'raison_sociale_x', 'raison_sociale_y',
    'date_mission_x', 'date_mission_y',
    'code_naf_x', 'code_naf_y',
    'libelle_naf_x', 'libelle_naf_y',
    # Scores RSE wide — redondants avec score_rse (format long) et score_rse_moy
    'score_global_2021', 'score_global_2022', 'score_global_2023', 'score_global_2024',
    # Variables calculées aberrantes présentes dans la BDD source
    'actif net au carré', 'roe calculé (%)',
]
panel.drop(columns=[c for c in DROP_COLS if c in panel.columns], inplace=True)

# Réorganisation : identité | RSE | financier brut | variables dérivées | reste
cols_id   = ['SIREN', 'raison_sociale', 'nom_commercial', 'code_naf', 'libelle_naf',
             'Secteur_EcoVadis', 'date_mission', 'date_creation', 'date_vote_mission', 'annee']
cols_rse  = ['score_rse', 'score_rse_moy']
cols_fin  = FIN_VARS
cols_norm = ['rentabilite_nette', 'tx_endettement', 'log_total_actif']
ordered   = cols_id + cols_rse + cols_fin + cols_norm
other_cols = [c for c in panel.columns if c not in ordered]
panel = (
    panel[[c for c in ordered if c in panel.columns] + other_cols]
    .sort_values(['SIREN', 'annee']).reset_index(drop=True)
)

print(f'Panel : {panel.shape[0]:,} observations × {panel.shape[1]} variables')
print(f'Entreprises : {panel["SIREN"].nunique():,} | Années : {ANNEES}')
panel.head(8)

Panel : 2,997 observations × 21 variables
Entreprises : 1,005 | Années : [2021, 2022, 2023, 2024]


,SIREN,nom_commercial,Secteur_EcoVadis,date_creation,date_vote_mission,annee,score_rse,score_rse_moy,ca,ca_export,resultat_net,fonds_propres,dettes_fin,gearing,roe_pct,roa_pct,ebit,total_actif,rentabilite_nette,tx_endettement,log_total_actif
0,300882115,SAINT BERNARD,Food & Beverage,1/1/1989,2022-07-28,2021,62.7000,62.3750,"14,000,000.0000",NaN,"490,000.0000",NaN,NaN,NaN,NaN,NaN,NaN,"6,000,000.0000",0.0817,NaN,15.6073
1,300882115,SAINT BERNARD,Food & Beverage,1/1/1989,2022-07-28,2022,59.8000,62.3750,"14,000,000.0000",NaN,"297,000.0000","3,910,000.0000",0.0000,-0.1000,7.6000,4.6000,"403,000.0000","6,000,000.0000",0.0495,0.0000,15.6073
2,300882115,SAINT BERNARD,Food & Beverage,1/1/1989,2022-07-28,2023,63.2000,62.3750,"15,100,000.0000",NaN,"-419,000.0000","3,490,000.0000","477,000.0000",0.0000,-12.0000,-5.6000,"-387,000.0000",NaN,NaN,NaN,NaN
3,300882115,SAINT BERNARD,Food & Beverage,1/1/1989,2022-07-28,2024,63.8000,62.3750,"15,000,000.0000",NaN,"-274,000.0000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,301852042,SEQUANO Amenagement,Construction,31/1/1975,2021-06-03,2021,59.4000,59.4500,"113,000,000.0000",0.0000,"161,000.0000","15,100,000.0000","64,200,000.0000",0.6000,1.1000,0.1000,"-1,330,000.0000","157,000,000.0000",0.0010,0.4089,18.8718
5,301852042,SEQUANO Amenagement,Construction,31/1/1975,2021-06-03,2022,58.1000,59.4500,"98,500,000.0000",0.0000,"420,000.0000","15,500,000.0000","68,700,000.0000",0.8000,2.7000,0.3000,"113,000.0000","129,000,000.0000",0.0033,0.5326,18.6753
6,301852042,SEQUANO Amenagement,Construction,31/1/1975,2021-06-03,2023,59.5000,59.4500,"86,300,000.0000",0.0000,"802,000.0000","21,800,000.0000","49,700,000.0000",0.0000,3.7000,0.6000,"-139,000.0000","126,000,000.0000",0.0064,0.3944,18.6518
7,303229504,AGROBIOTHERS LABORATOIRE,Food & Beverage,1/1/1975,2021-12-30,2021,62.7000,62.3750,"63,700,000.0000",0.0000,"1,460,000.0000","21,200,000.0000","5,050,000.0000",-0.4000,6.9000,3.8000,"2,020,000.0000","39,000,000.0000",0.0374,0.1295,17.4791


## 5. Diagnostic qualité

In [6]:
missing = (panel.isna().mean() * 100).round(1).sort_values(ascending=False)
print('Taux de valeurs manquantes par variable :')
print(missing[missing > 0].to_string())
print(f'\nObs. avec score_rse ET rentabilite_nette renseignés : '
      f'{panel.dropna(subset=["score_rse","rentabilite_nette"]).shape[0]:,}')

Taux de valeurs manquantes par variable :
ca_export           46.4000
ca                  40.4000
roa_pct             38.4000
roe_pct             38.4000
ebit                37.9000
tx_endettement      22.8000
rentabilite_nette   16.3000
log_total_actif     16.0000
total_actif         16.0000
nom_commercial      11.8000
dettes_fin           9.9000
gearing              3.9000
fonds_propres        2.6000
resultat_net         0.5000

Obs. avec score_rse ET rentabilite_nette renseignés : 2,508


In [7]:
panel[['score_rse', 'rentabilite_nette', 'tx_endettement', 'gearing', 'log_total_actif']].describe().round(3)

,score_rse,rentabilite_nette,tx_endettement,gearing,log_total_actif
count,"2,997.0000","2,508.0000","2,313.0000","2,881.0000","2,516.0000"
mean,57.8370,1.4720,11.2820,-0.7050,15.5990
std,4.6000,76.1700,525.7540,82.5140,2.3380
min,45.5000,-25.3680,0.0000,"-1,540.0000",6.0570
25%,57.7000,-0.0280,0.0840,-0.7000,13.8160
50%,58.3000,0.0240,0.2240,-0.1000,15.4250
75%,61.0000,0.0850,0.4260,0.6000,16.8600
max,66.6000,"3,814.2860","25,285.7140","2,910.0000",24.9860


## 6. Export

In [8]:
panel.to_csv('panel_rse_financier.csv', index=False)
print(f'Export OK : {panel.shape[0]:,} lignes × {panel.shape[1]} colonnes')

Export OK : 2,997 lignes × 21 colonnes
